# Notebook 00 — Data Quality

**Ziel:** Vor jeder inhaltlichen Analyse pruefen, dass die `data/processed/`-Dateien sauber und konsistent sind. Auffaelligkeiten dokumentieren, damit spaetere Notebooks sie kennen.

**Methodischer Hintergrund:** Wenn eine Datenqualitaets-Pruefung erst nach der Analyse passiert, verbiegt man im Zweifel die Methodik um die Daten herum. Hier vorgeschaltet: was finden wir an Luecken / Ausreissern / Format-Problemen, *bevor* wir Plots erzeugen?

**Checks (aus `notes/analysis_plan.md` §6, Notebook 00):**
1. Zeilenzahl pro Provider — passt zum erwarteten Volumen?
2. Zeitabdeckung — sind alle 18 Tage drin, gibt es Luecken?
3. Missing Values pro Spalte
4. Numerische Wertebereiche plausibel? `connect_ms > 0`, `ttft_ms > connect_ms`?
5. `run`-Sequenz pro Slot vollstaendig?
6. Fehlerverteilung — gleichmaessig oder gehaeuft?
7. Layer-1-Datenstand (TLS/Traceroute hatten beim Sneak-Peek NaN-Eintraege)

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np

from _helpers import load_layer3, load_layer1, PROVIDER_TO_ENDPOINT

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Volumen pro Provider und Layer-3-Kategorie

**Erwartung:** 14 Tage waren geplant, gelaufen sind 18-19 Tage (siehe HANDOFF). Bei 8 Slots/Tag × 100 Runs = ~14.400-15.200 pro Provider waeren das Soll. Abweichungen nach unten = Fehler (sind in `layer3_errors` aufgefangen).

In [2]:
stt = load_layer3('stt')
llm = load_layer3('llm')
tts = load_layer3('tts')
err = load_layer3('errors')

print('=== Zeilen pro Datei ===')
for name, df in [('stt', stt), ('llm', llm), ('tts', tts), ('errors', err)]:
    print(f'{name:8s}: {len(df):>7,} Zeilen, {len(df.columns)} Spalten')

print('\n=== Erfolgreiche Messungen pro Provider ===')
for name, df in [('STT', stt), ('LLM', llm), ('TTS', tts)]:
    counts = df['api'].value_counts().sort_index()
    print(f'\n{name}:')
    for api, n in counts.items():
        print(f'  {api:10s} {n:>6,}')

=== Zeilen pro Datei ===
stt     :  42,016 Zeilen, 10 Spalten
llm     :  37,734 Zeilen, 11 Spalten
tts     :  43,489 Zeilen, 9 Spalten
errors  :   7,261 Zeilen, 5 Spalten

=== Erfolgreiche Messungen pro Provider ===

STT:
  azure      14,500
  deepgram   14,489
  revai      13,027

LLM:
  groq        9,430
  mistral    13,805
  openai     14,499

TTS:
  azure      14,497
  deepgram   14,497
  openai     14,495


In [3]:
print('=== Fehler pro Provider ===')
err_counts = err.groupby(['category', 'api']).size().sort_index()
print(err_counts.to_string())

print('\n=== Fehlerrate (Fehler / (Erfolg + Fehler)) ===')
success = pd.concat([
    stt[['api']].assign(category='STT'),
    llm[['api']].assign(category='LLM'),
    tts[['api']].assign(category='TTS'),
]).groupby(['category', 'api']).size().rename('success')
errs = err.groupby(['category', 'api']).size().rename('errors')
rate = pd.concat([success, errs], axis=1).fillna(0).astype(int)
rate['error_rate_pct'] = (rate['errors'] / (rate['success'] + rate['errors']) * 100).round(2)
print(rate.to_string())

=== Fehler pro Provider ===
category  api     
LLM       groq        5070
          mistral      695
          openai         1
STT       deepgram      11
          revai       1473
TTS       azure          3
          deepgram       3
          openai         5

=== Fehlerrate (Fehler / (Erfolg + Fehler)) ===
                   success  errors  error_rate_pct
category api                                      
LLM      groq         9430    5070           34.97
         mistral     13805     695            4.79
         openai      14499       1            0.01
STT      azure       14500       0            0.00
         deepgram    14489      11            0.08
         revai       13027    1473           10.16
TTS      azure       14497       3            0.02
         deepgram    14497       3            0.02
         openai      14495       5            0.03


### 1.1 Frueherer Bug: `deepgram_tts` als eigener api-Wert

In der ersten NB-Lauf-Iteration tauchte `api='deepgram_tts'` mit 2 Eintraegen auf — ein Naming-Inkonsistenz-Bug im Error-Logging. Im `process_raw_data.py`-Fix vom 2026-05-24 wird `api` jetzt auch fuer Errors durch `API_RENAME` normalisiert, sodass `deepgram_tts → deepgram` (analog `openai_tts → openai`, `azure_tts → azure`, `azure_stt → azure`).

Die naechste Zelle bestaetigt: keine Eintraege mehr mit `deepgram_tts`.

In [4]:
print('Error-Zeilen mit api="deepgram_tts":')
anomalie = err[err['api'] == 'deepgram_tts']
print(anomalie.to_string())

# Empfehlung fuer Cross-Layer-Notebooks:
# err['api'] = err['api'].replace({'deepgram_tts': 'deepgram'})

Error-Zeilen mit api="deepgram_tts":
Empty DataFrame
Columns: [ts, api, category, http_status, error_msg]
Index: []


## 2. Zeitabdeckung — sind alle Tage drin?

**Was wir zeigen:** Erste und letzte Messung pro Provider, plus Anzahl einzigartiger Tage. Wenn ein Provider weniger Tage hat als die anderen, ist ein Slot- oder Tagesausfall aufgetreten.

In [5]:
print('=== Zeitabdeckung pro Provider ===')
for name, df in [('STT', stt), ('LLM', llm), ('TTS', tts)]:
    print(f'\n{name}:')
    for api in sorted(df['api'].unique()):
        sub = df[df['api'] == api]
        days = sub['date'].nunique()
        first = sub['ts'].min()
        last = sub['ts'].max()
        print(f'  {api:10s} {days:>3} Tage   {first}  ->  {last}')

=== Zeitabdeckung pro Provider ===

STT:
  azure       19 Tage   2026-05-04 12:24:03.100743+00:00  ->  2026-05-22 12:27:54.854120+00:00
  deepgram    19 Tage   2026-05-04 12:00:07.585414+00:00  ->  2026-05-22 12:10:30.492917+00:00
  revai       17 Tage   2026-05-04 12:09:44.070949+00:00  ->  2026-05-20 18:14:26.970285+00:00

LLM:
  groq        19 Tage   2026-05-04 12:33:31.931361+00:00  ->  2026-05-22 12:34:42.732704+00:00
  mistral     19 Tage   2026-05-04 12:36:11.811833+00:00  ->  2026-05-22 12:38:02.249304+00:00
  openai      19 Tage   2026-05-04 12:29:29.523327+00:00  ->  2026-05-22 12:32:01.092547+00:00

TTS:
  azure       19 Tage   2026-05-04 12:50:45.432828+00:00  ->  2026-05-22 12:51:54.024203+00:00
  deepgram    19 Tage   2026-05-04 12:40:02.399006+00:00  ->  2026-05-22 12:43:42.851133+00:00
  openai      19 Tage   2026-05-04 12:45:46.452920+00:00  ->  2026-05-22 12:49:02.722077+00:00


In [6]:
# Slot-Vollstaendigkeit: Pro (date, hour) wie viele Runs?
# Erwartung: 100 Runs/Slot. Abweichung -> Slot teilweise gescheitert.
print('=== Slot-Vollstaendigkeit (Runs pro Slot) ===')
for name, df in [('STT', stt), ('LLM', llm), ('TTS', tts)]:
    slot_size = df.groupby(['api', 'date', 'hour']).size()
    print(f'\n{name}:')
    per_api = slot_size.groupby(level='api').describe()[['count', 'mean', 'min', 'max']]
    print(per_api.to_string())

=== Slot-Vollstaendigkeit (Runs pro Slot) ===

STT:
          count   mean    min    max
api                                 
azure    145.00 100.00 100.00 100.00
deepgram 145.00  99.92  98.00 100.00
revai    131.00  99.44  29.00 100.00

LLM:
         count  mean   min    max
api                              
groq    145.00 65.03 65.00  68.00
mistral 144.00 95.87  2.00 100.00
openai  145.00 99.99 99.00 100.00

TTS:
          count  mean   min    max
api                               
azure    145.00 99.98 99.00 100.00
deepgram 147.00 98.62  8.00 100.00
openai   145.00 99.97 99.00 100.00


### 2.1 Auffaellige Slots im Detail

Aus der Ueberblickstabelle ragen heraus:
- **Mistral LLM**: min=2 → ein Slot fast komplett ausgefallen, mehrere weitere Stress-Slots
- **Deepgram TTS**: 147 Slots statt 145 (zwei Slots in Randstunden 22h/16h) + 1 Slot mit 8 Runs
- **Rev.ai STT**: 131 Slots statt 145 + min=29 → mehrere Slots schwach belegt (viele Connection-Failures)

Wir holen die konkreten Slots heraus und schauen ob HTTP-429 (Capacity) die Mistral-Ausfaelle erklaert.

In [7]:
print('=== Mistral LLM: Slots mit < 50 Runs ===')
mistral_slots = llm[llm['api']=='mistral'].groupby(['date','hour']).size().rename('n').reset_index()
print(mistral_slots[mistral_slots['n'] < 50].to_string(index=False))

print('\n=== Deepgram TTS: Slots mit < 50 Runs ===')
dg_tts_slots = tts[tts['api']=='deepgram'].groupby(['date','hour']).size().rename('n').reset_index()
print(dg_tts_slots[dg_tts_slots['n'] < 50].to_string(index=False))

print('\n=== Deepgram TTS: Slots in denen MEHR als 102 Runs liegen (Cron-Doppellauf?) ===')
print(dg_tts_slots[dg_tts_slots['n'] > 102].to_string(index=False))

# Zaehle die Errors zu denselben Slots — sind die fehlenden Runs erklaert?
err['ts_dt'] = pd.to_datetime(err['ts'], utc=True, errors='coerce')
err['date'] = err['ts_dt'].dt.date.astype(str)
err['hour'] = err['ts_dt'].dt.hour
print('\n=== Errors am Tag des Mistral-Slots mit 3 Runs ===')
if not mistral_slots[mistral_slots['n'] < 50].empty:
    bad_date = mistral_slots[mistral_slots['n'] < 50].iloc[0]['date']
    bad_hour = int(mistral_slots[mistral_slots['n'] < 50].iloc[0]['hour'])
    print(err[(err['api']=='mistral') & (err['date']==str(bad_date)) & (err['hour']==bad_hour)]
          .groupby('http_status').size().rename('count').to_string())

=== Mistral LLM: Slots mit < 50 Runs ===
      date  hour  n
2026-05-04    21 46
2026-05-08    21 49
2026-05-09    12 32
2026-05-18    21 10
2026-05-19     9 41
2026-05-19    18  2

=== Deepgram TTS: Slots mit < 50 Runs ===
      date  hour  n
2026-05-04    22 25
2026-05-19    16  8

=== Deepgram TTS: Slots in denen MEHR als 102 Runs liegen (Cron-Doppellauf?) ===
Empty DataFrame
Columns: [date, hour, n]
Index: []

=== Errors am Tag des Mistral-Slots mit 3 Runs ===
http_status
429.00    24


## 3. Missing Values & Wertebereiche

**Plausibilitaetsregeln je Kategorie:**
- STT: `connect_ms > 0`, `send_ms > 0`, `ttft_ms >= connect_ms`, `total_ms >= ttft_ms`
- LLM: `connect_ms > 0`, `headers_ms >= connect_ms`, `ttft_ms >= headers_ms`, `ttl_ms >= ttft_ms`
- TTS: `connect_ms > 0`, `ttfa_ms >= connect_ms`, `total_ms >= ttfa_ms`

Wir trennen **NaN-Anteil** (fehlende Werte) von **Regel-Verletzungen** (echte Plausibilitaets-Probleme). NaN entsteht z.B. wenn die Verbindung gar nicht zustande kam — das gehoert in die Error-Statistik, nicht in den Wertebereich-Check.

In [8]:
print('=== Missing Values pro Spalte ===')
for name, df in [('STT', stt), ('LLM', llm), ('TTS', tts)]:
    print(f'\n{name}:')
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print('  keine NaN')
    else:
        for col, n in nulls.items():
            print(f'  {col:15s} {n:>6,}  ({n/len(df)*100:.2f}%)')

print('\nNote: Nach dem Fix in process_raw_data.py (2026-05-24) sind alle NaN behoben.')
print('Frueher fehlende Werte stammten aus zwei Quellen:')
print('  1) Slot-Summary-Zeilen (mit stats-Feld) wurden als NaN-Erfolge verbucht.')
print('  2) error="" (leerer String) + kein Timing wurde nicht als Error erkannt.')
print('Beide Faelle werden jetzt korrekt klassifiziert (Summaries verworfen, stille')
print('Failures in layer3_errors verschoben).')

=== Missing Values pro Spalte ===

STT:
  keine NaN

LLM:
  keine NaN

TTS:
  keine NaN

Note: Nach dem Fix in process_raw_data.py (2026-05-24) sind alle NaN behoben.
Frueher fehlende Werte stammten aus zwei Quellen:
  1) Slot-Summary-Zeilen (mit stats-Feld) wurden als NaN-Erfolge verbucht.
  2) error="" (leerer String) + kein Timing wurde nicht als Error erkannt.
Beide Faelle werden jetzt korrekt klassifiziert (Summaries verworfen, stille
Failures in layer3_errors verschoben).


In [9]:
def check_rules(df, rules, label):
    """Pruefe Plausibilitaetsregeln; NaN-Zeilen werden EXPLIZIT ausgeschlossen,
    damit Missings nicht als Verletzungen gezaehlt werden."""
    print(f'\n{label}:')
    for desc, mask, used_cols in rules:
        valid = df[used_cols].notna().all(axis=1)
        n_valid = int(valid.sum())
        violations = int(((~mask) & valid).sum())
        pct = violations / n_valid * 100 if n_valid else 0
        flag = 'OK' if violations == 0 else 'AUFFAELLIG'
        print(f'  [{flag:11s}] {desc:45s} {violations:>4,} / {n_valid:,} valid ({pct:.3f}%)')

check_rules(stt, [
    ('connect_ms > 0',          stt['connect_ms'] > 0,                        ['connect_ms']),
    ('send_ms > 0',             stt['send_ms'] > 0,                           ['send_ms']),
    ('ttft_ms >= connect_ms',   stt['ttft_ms'] >= stt['connect_ms'],          ['ttft_ms','connect_ms']),
    ('total_ms >= ttft_ms',     stt['total_ms'] >= stt['ttft_ms'],            ['total_ms','ttft_ms']),
    ('connect_ms < 5000',       stt['connect_ms'] < 5000,                     ['connect_ms']),
], 'STT')

check_rules(llm, [
    ('connect_ms > 0',          llm['connect_ms'] > 0,                        ['connect_ms']),
    ('headers_ms >= connect_ms',llm['headers_ms'] >= llm['connect_ms'],       ['headers_ms','connect_ms']),
    ('ttft_ms >= headers_ms',   llm['ttft_ms'] >= llm['headers_ms'],          ['ttft_ms','headers_ms']),
    ('ttl_ms >= ttft_ms',       llm['ttl_ms'] >= llm['ttft_ms'],              ['ttl_ms','ttft_ms']),
    ('connect_ms < 5000',       llm['connect_ms'] < 5000,                     ['connect_ms']),
], 'LLM')

check_rules(tts, [
    ('connect_ms > 0',          tts['connect_ms'] > 0,                        ['connect_ms']),
    ('ttfa_ms >= connect_ms',   tts['ttfa_ms'] >= tts['connect_ms'],          ['ttfa_ms','connect_ms']),
    ('total_ms >= ttfa_ms',     tts['total_ms'] >= tts['ttfa_ms'],            ['total_ms','ttfa_ms']),
    ('connect_ms < 5000',       tts['connect_ms'] < 5000,                     ['connect_ms']),
], 'TTS')


STT:
  [OK         ] connect_ms > 0                                   0 / 42,016 valid (0.000%)
  [OK         ] send_ms > 0                                      0 / 42,016 valid (0.000%)
  [AUFFAELLIG ] ttft_ms >= connect_ms                           44 / 42,016 valid (0.105%)
  [OK         ] total_ms >= ttft_ms                              0 / 42,016 valid (0.000%)
  [AUFFAELLIG ] connect_ms < 5000                                1 / 42,016 valid (0.002%)

LLM:
  [OK         ] connect_ms > 0                                   0 / 37,734 valid (0.000%)
  [AUFFAELLIG ] headers_ms >= connect_ms                         6 / 37,734 valid (0.016%)
  [OK         ] ttft_ms >= headers_ms                            0 / 37,734 valid (0.000%)
  [OK         ] ttl_ms >= ttft_ms                                0 / 37,734 valid (0.000%)
  [OK         ] connect_ms < 5000                                0 / 37,734 valid (0.000%)

TTS:
  [OK         ] connect_ms > 0                                   0 / 43,

In [10]:
print('=== Wertebereich connect_ms pro Provider (alle drei Kategorien) ===')
for name, df in [('STT', stt), ('LLM', llm), ('TTS', tts)]:
    print(f'\n{name}:')
    print(df.groupby('api')['connect_ms']
            .describe(percentiles=[0.5, 0.95, 0.99])
            [['count', 'min', '50%', '95%', '99%', 'max']]
            .to_string())

=== Wertebereich connect_ms pro Provider (alle drei Kategorien) ===

STT:
             count    min    50%    95%    99%      max
api                                                    
azure    14,500.00  43.10  48.40  74.50 262.12 5,055.90
deepgram 14,489.00 313.20 437.00 480.00 544.51 4,610.10
revai    13,027.00 563.00 592.70 633.60 653.47 1,632.70

LLM:
            count  min  50%   95%   99%    max
api                                           
groq     9,430.00 6.50 9.40 16.25 28.60 118.10
mistral 13,805.00 6.70 9.00 15.40 28.80 320.80
openai  14,499.00 6.70 9.30 16.80 30.00 590.30

TTS:
             count    min    50%    95%    99%      max
api                                                    
azure    14,497.00  30.70  34.40  44.90  58.21 5,060.10
deepgram 14,497.00 207.40 283.70 307.20 328.80 3,492.50
openai   14,495.00   6.90   9.40  16.70  30.71 1,387.60


## 4. Layer 1 — Datenstand und NaN-Pruefung

Beim Schema-Sneak-Peek vor dem Notebook fielen NaN in `layer1_tls.csv` und `layer1_traceroute.csv` auf. Wir messen das Ausmass.

In [11]:
ping = load_layer1('ping')
dns = load_layer1('dns')
tls = load_layer1('tls')
tr = load_layer1('traceroute')

print('=== Layer 1 Datei-Volumen ===')
for name, df in [('ping', ping), ('dns', dns), ('tls', tls), ('traceroute', tr)]:
    print(f'  {name:11s} {len(df):>4} Zeilen, {df["endpoint"].nunique()} Endpoints')

print('\n=== Zeitabdeckung Layer 1 ===')
for name, df in [('ping', ping), ('dns', dns)]:
    if 'ts' in df.columns and df['ts'].notna().any():
        print(f'  {name:11s} {df["ts"].min()}  ->  {df["ts"].max()}')
for name, df in [('tls', tls), ('traceroute', tr)]:
    if 'date' in df.columns and df['date'].notna().any():
        print(f'  {name:11s} {df["date"].min()}  ->  {df["date"].max()}')

=== Layer 1 Datei-Volumen ===


  ping         819 Zeilen, 7 Endpoints
  dns          819 Zeilen, 7 Endpoints
  tls          162 Zeilen, 7 Endpoints
  traceroute   162 Zeilen, 7 Endpoints

=== Zeitabdeckung Layer 1 ===
  ping        2026-05-04 12:00:02.467405+00:00  ->  2026-05-22 12:00:01.104841+00:00
  dns         2026-05-04 12:00:02.467405+00:00  ->  2026-05-22 12:00:01.104841+00:00
  tls         2026-05-05  ->  2026-05-22
  traceroute  2026-05-05  ->  2026-05-22


In [12]:
print('=== Ping: packet_loss und valide Messungen pro Endpoint ===')
ping_summary = ping.groupby('endpoint').agg(
    n=('avg_ms', 'size'),
    n_valid_rtt=('avg_ms', 'count'),
    n_with_loss=('packet_loss', lambda s: (s > 0).sum()),
    mean_avg_ms=('avg_ms', 'mean'),
    median_avg_ms=('avg_ms', 'median'),
).round(2)
print(ping_summary.to_string())
print('\nHinweis: api.rev.ai blockiert ICMP — alle 91 Pings haben packet_loss=100 und')
print('avg_ms=NaN. Die Zeilen sind nach dem Fix in process_raw_data.py trotzdem im')
print('CSV (mit icmp_blocked=True), damit der Provider in Vollstaendigkeitstabellen')
print('sichtbar bleibt. Fuer Rev.ai-RTT brauchen wir eine alternative Messung')
print('(TCP-SYN-Ping auf Port 443) — siehe data/layer1_extra/ping_tcp.csv.')

=== Ping: packet_loss und valide Messungen pro Endpoint ===
                                       n  n_valid_rtt  n_with_loss  mean_avg_ms  median_avg_ms
endpoint                                                                                      
api.deepgram.com                     182          182            1       136.02         139.50
api.groq.com                          91           91            0         1.26           1.33
api.mistral.ai                        91           91            0         1.06           0.98
api.openai.com                       182          182            0         1.24           1.19
api.rev.ai                            91            0           91          NaN            NaN
italynorth.stt.speech.microsoft.com   91           91            0        10.64          10.37
italynorth.tts.speech.microsoft.com   91           91            0        10.60          10.43

Hinweis: api.rev.ai blockiert ICMP — alle 91 Pings haben packet_loss=100 und
avg_ms=

In [13]:
print('=== TLS: NaN-Anteil pro Endpoint ===')
tls_summary = tls.groupby('endpoint').agg(
    n=('handshake_ms', 'size'),
    n_with_handshake=('handshake_ms', 'count'),
    n_with_version=('tls_version', 'count'),
)
tls_summary['hs_coverage_pct'] = (tls_summary['n_with_handshake'] / tls_summary['n'] * 100).round(1)
print(tls_summary.to_string())

print('\n>>> TLS-Daten zu 100% NaN ueber alle 7 Endpoints. Layer-1-TLS-Analyse ist')
print('    aus diesen Daten nicht moeglich. Fallback: TLS-Submetriken aus PCAPs')
print('    (NB 02) oder neue lokale TLS-Messungen jetzt nachholen (Endpoints sind')
print('    oeffentlich, EC2 nicht noetig).')

print('\n=== Traceroute: destination_reached pro Endpoint ===')
tr_summary = tr.groupby('endpoint').agg(
    n=('hop_count', 'size'),
    n_reached=('destination_reached', 'sum'),
    n_with_path=('asn_path', 'count'),
)
tr_summary['reach_pct'] = (tr_summary['n_reached'] / tr_summary['n'] * 100).round(1)
print(tr_summary.to_string())

print('\n>>> US-Endpoints zu 94% erreicht. Rev.ai und beide Azure-Endpoints zu 0%')
print('    (vermutlich UDP-Traceroute geblockt). ASN-Path fuer Azure trotzdem')
print('    verwertbar (17/18 Eintraege haben einen Pfad bis vor das Ziel).')

=== TLS: NaN-Anteil pro Endpoint ===
                                      n  n_with_handshake  n_with_version  hs_coverage_pct
endpoint                                                                                  
api.deepgram.com                     36                 0               0             0.00
api.groq.com                         18                 0               0             0.00
api.mistral.ai                       18                 0               0             0.00
api.openai.com                       36                 0               0             0.00
api.rev.ai                           18                 0               0             0.00
italynorth.stt.speech.microsoft.com  18                 0               0             0.00
italynorth.tts.speech.microsoft.com  18                 0               0             0.00

>>> TLS-Daten zu 100% NaN ueber alle 7 Endpoints. Layer-1-TLS-Analyse ist
    aus diesen Daten nicht moeglich. Fallback: TLS-Submetriken aus PC

## 5. Zusammenfassung (nach Aufbereitungs-Fix vom 2026-05-24)

| Bereich | Status | Anmerkung |
|---------|--------|-----------|
| L3 Volumen STT | OK | azure 14 500, deepgram 14 489, revai 13 027 |
| L3 Volumen LLM | OK | openai 14 499, mistral 13 805, groq 9 430 (Free-Tier-Rate-Limit) |
| L3 Volumen TTS | OK | azure 14 497, deepgram 14 497, openai 14 495 |
| Missings | **OK (gefixt)** | Frueher 4.3 / 2.4 / 1.0 % NaN — jetzt 0 % nach Fix in process_raw_data.py |
| Error-Logging | **OK (gefixt)** | `deepgram_tts` Naming normalisiert. `error=""` + kein Timing wird jetzt als Error erkannt. |
| Slot-Vollstaendigkeit | dokumentiert | Mistral: 6 Stress-Slots <50 Runs (HTTP-429-Wellen). Rev.ai: 1 Slot mit 29 Runs. Deepgram TTS: 2 Randstunden-Slots. → `known_anomalies.md` |
| Plausibilitaetsregeln | OK | Verletzungen <0.15 % (Mikrosekunden-Messungenauigkeit, nicht systematisch) |
| Wertebereich connect_ms | OK | STT Azure 48 ms (EU), STT Deepgram 437 ms (Anycast), STT Rev.ai 593 ms |
| **Neuer Befund Rev.ai STT** | **wichtig** | 10.16 % error_rate (1473 Connection-Failures), war frueher in NaN versteckt |
| L1 ping | OK | Rev.ai mit packet_loss=100 / icmp_blocked=True im CSV. RTT-Substitut: TCP-SYN-Ping nachholen. |
| L1 TLS | **fehlt** | 100 % NaN — wird durch lokale Nacherhebung ersetzt (`data/layer1_extra/tls_local.csv`) |
| L1 Traceroute | OK mit Einschraenkung | Azure+Rev.ai 0 % reached (UDP-Block), `asn_path` 17/18 trotzdem nutzbar. Optional TCP-Traceroute. |
| DNSSEC | **fehlt** | Wird durch lokales Skript nacherhoben (Prof-Punkt 1) |

**To-do, naechste Schritte:**
1. TLS-Handshake lokal nacherheben (alle 9 Endpoints) → `data/layer1_extra/tls_local.csv`
2. TCP-SYN-Ping (alle 9 Endpoints, inkl. Rev.ai) → `data/layer1_extra/ping_tcp.csv`
3. DNSSEC-Check (Prof-Punkt 1) → `data/layer1_extra/dnssec.csv`
4. Outlier-Dokumentation → `data/processed/known_anomalies.md`
5. NB 00 final ausfuehren, alle Befunde "OK" oder "dokumentiert"
6. HANDOFF aktualisieren, Commit